# Streaming & Real-TimePatterns

**Module 7 · Lesson 7.2**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- SDK Streaming - All Three Providers
- FastAPI SSE Endpoint - Stream to Frontend
- WebSocket Bridge - Full-Duplex Chat
- Deploy to Cloud Run - Production Streaming
- TTFB & Provider Streaming Formats - OpenAI vs Anthropic vs Gemini
- Frontend Streaming - React, Vercel AI SDK, Markdown Rendering

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install "anthropic>=0.40.0" openai google-genai python-dotenv websockets -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


## Demo Day, 4 PM: The Stream That Arrived As A Parcel

At 9 AM this endpoint streamed beautifully on your laptop. At the 4 PM stakeholder demo, on staging, the same code did this:

**📝 `demo_day.txt`** - *Intro*

In [ ]:
# Output: 9:00 AM - your laptop:
# $ curl -N localhost:8000/chat/stream -d '{"prompt":"..."}'
#   data: {"token": "The"}        <- 210ms
#   data: {"token": " capital"}   <- 251ms   ...a new token every ~40ms. Perfect.
#
# 4:00 PM - staging, behind nginx, in front of your stakeholders:
# $ curl -N https://staging.yourapp.com/chat/stream -d '{"prompt":"..."}'
#
#   [ ............ 28 seconds of nothing ............ ]
#
#   data: {"token": "The"} data: {"token": " capital"} ... ALL 412 tokens, one blob.
#
# Your PM: "I thought you said it streams?"
# Your code did not change. The proxy between you and the user did.

Streaming is not a Python feature - it is an agreement between your generator, your framework, every proxy in the middle, and the browser. Any one of them can silently turn your conveyor belt back into a parcel. This lesson builds the belt end to end (SDK → FastAPI → the wire → the browser), then walks the production path where it breaks. By Step 7 you will know exactly which layer buffered you - and the two lines that fix it.

> 📦 **Info**
>
> 🔧 Before you start: three commands and four words
> - **Install and run:** `pip install fastapi uvicorn "anthropic>=0.40.0" openai google-genai`, start servers with `uvicorn main:app --reload`, and test streams with `curl -N` (the `-N` disables curl's own buffering - yes, even curl buffers).
> - **90-second async primer:** `async def` declares a coroutine, `await` yields control while I/O happens, and everything shares ONE event loop - so a blocking (non-await) call inside an async function freezes every other request on the server. That one rule explains half this lesson's code.
> - **TTFT** = time to first *token* (the UX metric this lesson optimizes). **TTFB** = time to first *byte* of any HTTP response (the general web metric). Provider benchmarks often say TTFB when they measure TTFT - this lesson uses TTFT for model output throughout.
> - **Keys:** same three environment variables as Lesson 7.1 (`OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GOOGLE_API_KEY`). Any one is enough to follow along.

Sixty-second warm-up - three cards from the lessons this one stands on. Tap each to flip it.

> 📦 **Analogy**
>
> **Non-streaming = Amazon delivery.** Wait 2 days, get everything at once. **Streaming = conveyor belt sushi.** Plates arrive one by one as the chef makes them. You start eating immediately. **TTFT (Time To First Token): 200ms vs 5 seconds. Same total meal, ~25x faster first bite.**
> **Where this analogy breaks down:** the chef cannot un-serve a plate - once a token is displayed there is no moderation recall, which is why streaming guardrails buffer a few plates before the belt (Step 8). And unlike a sushi bar, a reverse proxy can secretly collect all your plates and deliver them as one stack (Step 7) - the belt only works if every hop cooperates.

Streaming in action - tokens arrive one by one:

Illustrative pacing: tokens roughly 30-50ms apart, first token typically ~200ms (sourced medians in Step 5).

> 💡 **Info**
>
> ⚠️ Common misconception
> You might think streaming makes the model *generate faster*. It does not - total generation time is identical (the comparison table below shows the same 2,000ms in both columns). Streaming only moves the first paint earlier and lets the user read while the model writes. And reasoning models invert the win: extended-thinking modes can spend tens of seconds before the FIRST token, so a "streaming" UX still needs a thinking indicator. The animation after Step 1 makes both points visible.

## Build the Stream

---

## 🎯 Concept 1: SDK Streaming - All Three Providers

### SDK Streaming - All Three Providers

Stream tokens from Gemini, OpenAI, Claude. Measure TTFT for each.

**📝 `01_sdk_streaming.py`** - *Streaming*

In [ ]:
# SDK STREAMING - all three providers on ASYNC clients
# pip install "anthropic>=0.40.0" openai google-genai
# (google-generativeai reached end-of-life 2025-11-30 - google-genai replaces it)
import asyncio, os, time

prompt = "Explain streaming in LLM APIs in 3 sentences."

async def timed(name, agen):
    # one timing harness for all three providers
    start = time.time(); ttft = None
    async for text in agen:
        if ttft is None: ttft = time.time() - start  # only the FIRST chunk sets it
        print(text, end="", flush=True)
    print(f"\n{name} TTFT: {ttft*1000:.0f}ms\n")

async def main():
    # Claude - stream() context manager, timeout + typed errors built in
    import anthropic
    cl = anthropic.AsyncAnthropic(timeout=30.0)
    async def claude():
        async with cl.messages.stream(model="claude-sonnet-4-6", max_tokens=200,
            messages=[{"role":"user","content":prompt}]) as st:
            async for t in st.text_stream: yield t
    await timed("Claude", claude())

    # OpenAI
    from openai import AsyncOpenAI
    oai = AsyncOpenAI(timeout=30.0)
    async def gpt():
        stream = await oai.chat.completions.create(model="gpt-5.4",
            messages=[{"role":"user","content":prompt}], stream=True)
        async for chunk in stream:
            if chunk.choices[0].delta.content: yield chunk.choices[0].delta.content
    await timed("OpenAI", gpt())

    # Gemini - the google-genai async client
    from google import genai
    g = genai.Client()  # reads GOOGLE_API_KEY
    async def gemini():
        stream = await g.aio.models.generate_content_stream(
            model="gemini-3.5-flash", contents=prompt)
        async for chunk in stream:
            if chunk.text: yield chunk.text
    await timed("Gemini", gemini())

asyncio.run(main())

# Output: (representative run; medians move - see the sourced table in Step 5)
# Streaming sends each token the moment the model produces it, so users read...
# Claude TTFT: 487ms
# Streaming lets an API return partial output while generation continues...
# OpenAI TTFT: 1096ms
# Streaming delivers tokens incrementally over one HTTP response...
# Gemini TTFT: 342ms
# Note: total generation time is unchanged - only the FIRST paint moved.

- `timed()` - one stopwatch for all three belts: it starts on request, stamps TTFT when the FIRST plate arrives (that is why only `ttft is None` sets it), and prints every token as it lands.

- `AsyncAnthropic().messages.stream(...)` - a context manager: it opens the SSE connection, `.text_stream` yields plain text deltas, and closing the `async with` block cleans up the connection even if you bail early.

- `AsyncOpenAI ... stream=True` - returns chunks whose `delta.content` is None on bookkeeping frames (role header, finish frame) - that is what the `if` guards.

- `genai.Client().aio` - the async surface of the google-genai SDK. The old google-generativeai SDK had no async client, which is exactly why the previous version of this lesson had to block the event loop.

- Every client gets `timeout=30.0` - a real network timeout that cancels a hung connection, not a post-hoc latency check.

Read it as: *three belts, one stopwatch - the differences are in the plumbing, not the concept.*

#### 💡 What just happened

⚡ What just happened? Three providers, three streaming APIs. Gemini: `generate_content_stream()`. OpenAI: `stream=True`. Claude: `messages.stream()`. All yield chunks; representative TTFT medians run ~340-1,100ms (sourced table in Step 5). **TTFT is the key UX metric - users start reading immediately instead of waiting for the full response.** The `UnifiedLLM` wrapper you built in Lesson 7.1 grows a `.stream()` method in this lesson's practice lab - same adapter idea, now per-token.

- **Scene 1 (standard model):** the same 2,000ms response rendered twice. Batch shows a spinner then a wall of text; streaming paints from ~200ms. Notice: both panels finish at the SAME instant - the total-time figure never differs. Streaming moves the wait, it does not shrink it.

- **Scene 2 (reasoning model):** extended-thinking modes delay even the first token - watch the thinking phase hold BOTH panels dark. Do: drag the response-length slider and switch provider TTFT presets to feel how first-paint dominates perceived speed.

Controls: Play / Reset / Speed (Slow / Normal / Fast); scene buttons or arrow keys; length slider; provider preset chips.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: FastAPI SSE Endpoint - Stream to Frontend

### FastAPI SSE Endpoint - Stream to Frontend

Build an API that streams LLM tokens via Server-Sent Events.

SSE Streaming Architecture

**📝 `02_fastapi_sse.py`** - *FastAPI*

In [ ]:
# FASTAPI SSE - stream LLM tokens to the browser
# run: uvicorn main:app --reload     test: curl -N -X POST localhost:8000/chat/stream -d '{"prompt":"hi"}'
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from google import genai
import json

app = FastAPI()
# scope CORS to YOUR frontend - a wildcard lets any website spend your tokens
app.add_middleware(CORSMiddleware, allow_origins=["https://yourapp.example"],
                   allow_methods=["POST"], allow_headers=["content-type"])
client = genai.Client()  # reads GOOGLE_API_KEY from the environment

def sse(event: str, data: dict) -> str:
    # the SSE wire format: an event name, a data line, and a BLANK line as the frame boundary
    return f"event: {event}\ndata: {json.dumps(data)}\n\n"

@app.post("/chat/stream")
async def chat_stream(request: Request):
    body = await request.json()
    async def gen():
        tokens = 0
        try:
            yield sse("status", {"state": "generating"})
            stream = await client.aio.models.generate_content_stream(
                model="gemini-3.5-flash", contents=body["prompt"])
            async for chunk in stream:
                if await request.is_disconnected():   # tab closed? stop generating, stop paying
                    print(f"client gone after {tokens} tokens"); return
                if chunk.text:
                    tokens += 1
                    yield sse("token", {"text": chunk.text})
            yield sse("done", {"tokens": tokens})
        except Exception as e:
            yield sse("error", {"message": str(e)})  # never die silently mid-stream
    return StreamingResponse(gen(), media_type="text/event-stream",
        headers={"Cache-Control":"no-cache", "X-Accel-Buffering":"no"})
# X-Accel-Buffering: no tells nginx / Cloud Run: do NOT buffer this response

# Output: (curl -N, representative)
# event: status
# data: {"state": "generating"}
# event: token
# data: {"text": "Streaming"}
# ...a new frame every ~40ms...
# event: done
# data: {"tokens": 42}

- `sse()` - the wire format itself: `event:` names the frame (so the client can route status vs token vs error), `data:` carries JSON, and the BLANK line is the frame boundary. Forget the blank line and the browser waits forever.

- `async def gen()` + the async google-genai client - every wait point is an `await`, so OTHER requests keep flowing while this one streams. The previous version of this lesson iterated a sync SDK here, which froze the entire event loop per chunk.

- `request.is_disconnected()` - the billing guard: when the tab closes, generation stops. Without it the model keeps producing (and charging) into a void.

- the `except` arm - a mid-stream provider error becomes an `event: error` frame the client can render, instead of a silently truncated answer.

Read it as: *name your frames, await your waits, and always plan for the closed tab.*

**📝 `chat.js`** - *Browser*

In [ ]:
%%javascript
// THE CONSUMER - fetch() + ReadableStream (EventSource cannot POST or send headers)
const controller = new AbortController();
stopBtn.onclick = () => controller.abort();      // the "Stop generating" button

const res = await fetch("/chat/stream", {
  method: "POST",
  headers: {"content-type": "application/json"},
  body: JSON.stringify({prompt}),
  signal: controller.signal,
});

const reader = res.body.pipeThrough(new TextDecoderStream()).getReader();
let buf = "";
while (true) {
  const {value, done} = await reader.read();
  if (done) break;
  buf += value;
  let idx;
  while ((idx = buf.indexOf("\n\n")) !== -1) {   // one SSE frame per blank line
    const frame = buf.slice(0, idx); buf = buf.slice(idx + 2);
    const ev = /^event: (.*)$/m.exec(frame)?.[1] ?? "message";
    const data = JSON.parse(/^data: (.*)$/m.exec(frame)[1]);
    if (ev === "token") output.textContent += data.text;
    if (ev === "error") showError(data.message);
  }
}
// >>> Output: tokens append to the page every ~40ms; Stop aborts the fetch,
// which triggers request.is_disconnected() server-side. Full circle.

#### 💡 What just happened

⚡ What just happened? The endpoint streams **named SSE events** (status / token / done / error) and the browser parses frames on the blank-line boundary - no EventSource, because this endpoint needs POST, a JSON body, and headers, which EventSource cannot do (exactly what the quiz above established). The Stop button and the server's disconnect check are two halves of ONE feature: cancel on the client, stop billing on the server. This endpoint is the skeleton that grows into a full chat application in Lesson 7.4.

---

## 🎯 Concept 3: WebSocket Bridge - Full-Duplex Chat

### WebSocket Bridge - Full-Duplex Chat

Client can send AND receive simultaneously. User can interrupt mid-generation.

**📝 `03_websocket.py`** - *WebSocket*

In [ ]:
# WEBSOCKET - full duplex WITH REAL interruption
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from google import genai
import asyncio, json

app = FastAPI()
client = genai.Client()

async def generate(ws: WebSocket, prompt: str):
    try:
        stream = await client.aio.models.generate_content_stream(
            model="gemini-3.5-flash", contents=prompt)
        async for chunk in stream:
            if chunk.text: await ws.send_json({"type":"token", "text":chunk.text})
        await ws.send_json({"type": "done"})
    except asyncio.CancelledError:
        await ws.send_json({"type": "cancelled"})  # confirm the interrupt to the client
        raise

@app.websocket("/ws/chat")
async def ws_chat(ws: WebSocket):
    await ws.accept()
    task = None
    try:
        while True:
            msg = json.loads(await ws.receive_text())   # ALWAYS listening - generation runs elsewhere
            if msg["type"] == "stop" and task and not task.done():
                task.cancel()                              # <- the actual interruption
            elif msg["type"] == "prompt":
                if task and not task.done(): task.cancel()  # new question replaces the old answer
                task = asyncio.create_task(generate(ws, msg["text"]))
    except WebSocketDisconnect:
        if task and not task.done(): task.cancel()
# The receive loop and the generation task run CONCURRENTLY.
# That concurrency - not the protocol - is what makes interruption possible.

# Output: (client sends {"type":"stop"} mid-generation)
# {"type":"token","text":"Server-sent"} {"type":"token","text":" events"} ...
# {"type":"cancelled"}
# ...and the very next prompt starts a fresh generation task immediately.

- `asyncio.create_task(generate(...))` - generation becomes a background task, so the `while True` receive loop keeps listening DURING generation. A naive handler that streams inside the receive loop can never hear "stop" - the message just queues in the socket buffer until the answer finishes.

- `task.cancel()` - raises `CancelledError` inside the generation task at its next `await`; the task's `except` arm confirms with a `cancelled` frame, then re-raises (swallowing CancelledError is an asyncio anti-pattern).

- new prompt while answering - cancel-then-replace gives "ask a follow-up without waiting", the actual UX reason to pay for WebSocket's complexity.

- `WebSocketDisconnect` - the closed-tab guard again: kill the orphaned generation task or it streams (and bills) into nowhere.

Read it as: *one ear always open, one mouth cancellable - duplex is a concurrency pattern, not just a protocol.*

#### 💡 What just happened

⚡ What just happened? This handler delivers what the step title promises: send `{"type":"stop"}` mid-generation and the token flow halts within one chunk. The cost is real - you now own task lifecycles, reconnection, and load-balancer stickiness that SSE gave you for free. Rule of thumb: SSE for one-way token delivery (this module's default), WebSocket only when the client must talk WHILE the server streams - interruption, voice, collaborative editing. We'll stream tool calls over this machinery in Lesson 7.5.

---

## 🎯 Concept 4: Deploy to Cloud Run - Production Streaming

### Deploy to Cloud Run - Production Streaming

Auto-scaling, HTTPS, global CDN. Critical flags for streaming to work.

**📝 `04_cloud_run_deploy.sh Cloud`** - *Run*

In [ ]:
%%bash
# (deployment/terminal commands - run in a shell, not Colab, unless configured)
# Cloud Run deployment for streaming API
# store the key in Secret Manager ONCE (never in the deploy command or env vars):
gcloud secrets create google-api-key --data-file=- <<< "$GOOGLE_API_KEY"

gcloud run deploy genai-streaming \
  --source . \
  --region asia-south1 \
  --memory 512Mi --cpu 1 \
  --min-instances 0 --max-instances 10 \
  --timeout 600 \
  --no-cpu-throttling \
  --set-secrets GOOGLE_API_KEY=google-api-key:latest
# add auth in front (IAM/ID tokens or an API key check) before real traffic -
# an open LLM endpoint is a token faucet for whoever finds the URL.

# Critical flags:
# --no-cpu-throttling  Keep CPU active while the response streams (NOTE: this switches
#                      billing to instance-time - the "scales to zero" cost story changes)
# --timeout 600        The DEFAULT is 300s - long generations need explicit headroom
# X-Accel-Buffering:no In the FastAPI response headers (Step 2) - prevents buffering
# --set-secrets        Key comes from Secret Manager, not shell history or the console

# Output: (representative)
# Deploying container to Cloud Run service [genai-streaming]...
# Service URL: https://genai-streaming-xxxxx-el.a.run.app
# curl -N -X POST $URL/chat/stream ... -> event frames arrive one by one, unbuffered

> 💡 **Info**
>
> Cloud Run streaming gotchas
> - **Buffering:** Add `X-Accel-Buffering: no` header. Without it, Cloud Run buffers the entire response.
> - **CPU throttling:** Use `--no-cpu-throttling` to keep CPU active during streaming.
> - **Timeout:** Default 300s. Set higher for long generations.
> - **WebSockets:** Supported natively on Cloud Run since 2021.

| Metric | Batch | Streaming | Improvement |
|---|---|---|---|
| TTFT (illustrative 2s response) | ~2000ms | ~150ms | ~13x |
| Perceived speed | Slow | Fast | First paint 13x earlier |
| Total time | 2000ms | 2000ms | Same |
| Implementation | Simple | SSE + fetch/ReadableStream | Medium |

This table's example is a 2-second response (150ms first paint = ~13x). The hero's 5-second example gives ~25x - the ratio scales with response length, which is exactly why streaming matters MORE for long outputs.

## Ship It: Formats, Frontends, and the Production Path

---

## 🎯 Concept 5: TTFB & Provider Streaming Formats - OpenAI vs Anthropic vs Gemini

### TTFB & Provider Streaming Formats - OpenAI vs Anthropic vs Gemini

Groq: ~150ms TTFT. Reasoning modes: up to ~67s. Each provider streams differently.

| Provider | TTFB | Output Speed | Event Format | Usage in Stream |
|---|---|---|---|---|
| Groq (Llama 3.3 70B) | ~120-180ms | ~330 t/s | OpenAI-compatible delta | Final chunk |
| OpenAI GPT-5.5 | ~1.1s | ~90 t/s | delta.content tokens | stream_options opt-in |
| Anthropic Claude Sonnet 4.6 | ~0.5s (P95 ~0.9s) | ~70-80 t/s | 7 event types lifecycle | message_start + message_delta |
| Google Gemini (3.x Flash) | sub-second | fast, chunky | Multi-sentence chunks | Every chunk (cumulative) |
| Reasoning modes (any provider) | seconds to ~67s | - | thinking precedes tokens | varies |

Medians from [artificialanalysis.ai provider leaderboard](https://artificialanalysis.ai/leaderboards/providers), captured 2026-07-02. These numbers move monthly - re-measure before you architect around them. The extreme case: extended-thinking modes can push TTFT to ~67 seconds - stream a thinking indicator, not a blank screen.

> ✅ **Info**
>
> 🔍 Provider event format differences
> - **OpenAI:** `choices[0].delta.content` = single token. First chunk has `role: "assistant"`. Last has `finish_reason: "stop"`. Tool calls stream via `delta.tool_calls[index].function.arguments`.
> - **Anthropic:** Rich lifecycle: `message_start → content_block_start → content_block_delta → content_block_stop → message_delta → message_stop`. Input tokens in `message_start`, output tokens in `message_delta`. Extended thinking adds `thinking_delta` events.
> - **Gemini:** Streams multi-sentence chunks (not individual tokens). Each chunk is a complete `GenerateContentResponse` with safety ratings. REST needs `?alt=sse`. Usage is cumulative in every chunk.

#### 💡 What just happened

What just happened? TTFT varies **~400x across configurations** - Groq at ~150ms vs extended-thinking modes at ~67s. Reasoning models (o3, extended thinking) add seconds-to-minutes before any visible tokens. The three providers use fundamentally different streaming formats: OpenAI streams individual token deltas, Anthropic uses a structured 7-event lifecycle, and Gemini sends multi-sentence chunks with cumulative metadata. For production: use LiteLLM (Lesson 7.1) to normalize these differences, or implement provider-specific parsers. The billing half of this story - caching discounts, batch tiers, and metering cancelled streams - is Lesson 7.3 (Cost Engineering). **Key metric:** sub-500ms TTFB feels instant for interactive apps.

- **Scene 1 (three lanes):** one identical answer streamed simultaneously in each provider's wire format. Do: click any event chip to inspect its raw JSON in the wire log. Notice: where usage lives - OpenAI only in a final opt-in chunk, Anthropic split across message_start (input) and message_delta (output), Gemini cumulative in every chunk.

- **Scene 2 (the cancelled stream):** hit Cancel at ~60% and check the scratchpad's "usage received?" list - OpenAI reports nothing, Anthropic only input tokens, Gemini the last cumulative snapshot. If you bill from provider-reported usage, this is your blind spot; estimate from accumulated text as fallback. Details in Lesson 7.3 (Cost Engineering).

Controls: Play / Reset / Speed (Slow / Normal / Fast); Cancel stream in scene 2; click any event chip to inspect.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Frontend Streaming - React, Vercel AI SDK, Markdown Rendering

### Frontend Streaming - React, Vercel AI SDK, Markdown Rendering

useChat hook. Typewriter effect. Streaming markdown. AbortController cancellation.

> 📦 **Info**
>
> 🎨 Frontend streaming patterns
> - **Vercel AI SDK v6** ([vercel.com/blog/ai-sdk-6](https://vercel.com/blog/ai-sdk-6), repo [github.com/vercel/ai](https://github.com/vercel/ai)): `useChat` hook handles streaming state, SSE consumption, multi-provider support. Backend: `streamText()` → `toUIMessageStreamResponse()`. AI Elements provides prebuilt React components.
> - **Performance:** Accumulate tokens in `useRef`, batch-update `useState` on `requestAnimationFrame`. React 18 `useTransition` marks streaming updates as non-urgent. Never setState on every single token.
> - **Markdown:** Standard parsers break on partial markdown. Solutions: **streaming-markdown** ([github.com/thetarnav/streaming-markdown](https://github.com/thetarnav/streaming-markdown), 3kB incremental parser), **llm-ui** (React, block-based), or deferred approach (raw text during stream → full markdown after completion).
> - **Cancellation:** `AbortController` with `fetch(url, {signal: controller.signal})`. Combined: `AbortSignal.any([userSignal, AbortSignal.timeout(30000)])` handles both user cancel and timeout.
> - **Typewriter effect:** CSS `animation: blink 0.7s step-end infinite` on a cursor element appended after streaming content. Remove cursor on stream completion.

#### 💡 What just happened

What just happened? The frontend streaming stack has standardized. **Vercel AI SDK** is the dominant framework - `useChat` handles SSE consumption, state management, and multi-provider routing in one hook. **Markdown rendering during streaming** is a solved-but-nuanced problem: use incremental parsers that handle partial fragments (streaming-markdown, llm-ui), or buffer and render after completion. **Always sanitize LLM output with DOMPurify** - treat it as user-generated content due to prompt injection risks. **AbortController** is essential for "Stop generating" buttons. We assemble these pieces into a complete chat application in Lesson 7.4 (AI Chat with FastAPI).

---

## 🎯 Concept 7: Production Hardening - Proxies, Errors, Observability

### Production Hardening - Proxies, Errors, Observability

nginx proxy_buffering off. Cloudflare buffers SSE. Langfuse for stream observability.

> 💡 **Info**
>
> ⚠️ The #1 streaming failure: reverse proxy buffering
> - **nginx:** Buffers by default! Add: `proxy_buffering off; proxy_http_version 1.1; proxy_read_timeout 86400s;` Backend must send `X-Accel-Buffering: no` header.
> - **Cloudflare:** Buffers SSE by default, accumulates ~100KB before flushing. Full streaming control is **Enterprise-only**. Workaround: use Cloudflare Workers with ReadableStream, or route streaming endpoints around Cloudflare.
> - **AWS API Gateway:** REST APIs gained SSE streaming in Nov 2025 via `responseTransferMode: "STREAM"` with up to 15-min timeouts.
> - **Heartbeats:** Send `: heartbeat` every 15 seconds to prevent proxy/load balancer timeouts.

#### 💡 What just happened

What just happened? Reverse proxy configuration is the **most common cause of streaming failures** in production. nginx and Cloudflare both buffer responses by default - your perfectly working dev environment breaks the moment you deploy behind a proxy. **Observability:** log metadata at stream start, completion data at stream end via `onFinish` callback. Langfuse v3 is OTEL-native with `gen_ai.*` attributes. Track four metrics: TTFB (<500ms target), tokens/second, stream completion rate, and connection duration. Rate limit streaming connections separately: max 3 concurrent streams per user + token budget per hour. Turning these stream metrics into real dashboards and traces is Lesson 7.6 (Observability with Langfuse).

- **Scene 1 (the buffer):** tokens flow LLM → FastAPI → proxy → browser. Do: flip the proxy's buffering toggle and watch tokens POOL inside the proxy while the browser lane starves - then flush in one burst. That burst is the cold-open's demo-day failure, mechanized. Notice: the timing histogram - a huge first gap then near-zero gaps is the buffering fingerprint your monitoring can detect.

- **Scene 2 (heartbeats):** an idle stream against a load-balancer timeout countdown. Turn heartbeat comments on and watch each `: heartbeat` frame reset the clock.

Controls: Play / Reset / Speed (Slow / Normal / Fast); buffering toggle; heartbeat toggle.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 8: Advanced Streaming - RAG, Tool Calls, Multi-Model Racing

### Advanced Streaming - RAG, Tool Calls, Multi-Model Racing

Stream while retrieving. Accumulate partial JSON for tool calls. Race providers.

> ✅ **Info**
>
> 🔗 Advanced streaming patterns
> - **RAG streaming:** Synchronous retrieval (~600ms) → stream generation. Use SSE event types: `event: status` ("Searching...") → `event: metadata` (sources) → `event: token` (streamed answer). Cohere provides inline citations during streaming.
> - **Tool call streaming:** Function names arrive first, arguments stream character by character. Accumulate JSON fragments per tool_call index. Parse complete JSON when `finish_reason: "tool_calls"`. Use `partial-json-parser` for progressive UI updates.
> - **Multi-model racing:** `asyncio.wait(tasks, return_when=FIRST_COMPLETED)` → cancel slower streams → return fastest. Doubles cost. Alternative: fallback (try primary, switch on error) is cheaper.
> - **Streaming guardrails:** Buffer ~20-50 tokens → moderate batch → release. NVIDIA NeMo Guardrails supports configurable chunk sizes. Adds ~500ms display delay but catches harmful content before display.

#### 💡 What just happened

What just happened? Advanced streaming adds complexity at every layer. **RAG streaming** uses SSE event types to show status progression before tokens flow. **Tool call streaming** requires careful JSON fragment accumulation - OpenAI interleaves parallel tool calls by index. **Streaming guardrails** create a fundamental tension: tokens reach users before the full response can be moderated. The practical compromise is sentence-level buffering with async moderation. **Multi-model racing** is powerful but expensive - fallback chains (Lesson 7.1) are more cost-effective for most use cases. Tool-call streaming gets its full treatment in Lesson 7.5 (Function Calling & Tools).

**📝 `08_multimodel_race.py`** - *Racing*

In [ ]:
# Race two providers, keep whoever streams the FIRST token, cancel the loser
import asyncio

async def first_token(name, agen_factory):
    try:
        async for text in agen_factory():
            return name, text   # first chunk wins
    except asyncio.CancelledError:
        raise
    except Exception as e:
        return name, f"[{name} failed: {e}]"

# Two async-generator FACTORIES - each returns a fresh stream when called.
# They reuse the exact SDK streaming patterns from Step 1.
import anthropic
from google import genai

PROMPT = "Explain streaming in one sentence."

async def claude_stream():
    cl = anthropic.AsyncAnthropic(timeout=30.0)
    async with cl.messages.stream(model="claude-sonnet-4-6", max_tokens=100,
        messages=[{"role": "user", "content": PROMPT}]) as st:
        async for t in st.text_stream:
            yield t

async def gemini_stream():
    stream = await genai.Client().aio.models.generate_content_stream(
        model="gemini-3.5-flash", contents=PROMPT)
    async for chunk in stream:
        if chunk.text:
            yield chunk.text

async def race(claude_factory, gemini_factory):
    tasks = [
        asyncio.create_task(first_token("claude", claude_factory)),
        asyncio.create_task(first_token("gemini", gemini_factory)),
    ]
    done, pending = await asyncio.wait(
        tasks, timeout=30.0, return_when=asyncio.FIRST_COMPLETED)
    for t in pending:
        t.cancel()   # stop the slower stream - you still pay for it
    return next(iter(done)).result()

winner, text = asyncio.run(race(claude_stream, gemini_stream))
print(winner, text)
# Output: (representative - needs live API keys; which provider's first token
#          arrives first varies by network and load, e.g.)
# gemini Streaming delivers tokens incrementally over one HTTP response...

---

## 🎯 Concept 9: Real-Time Audio - OpenAI Realtime API & Gemini Live

### Real-Time Audio - OpenAI Realtime API & Gemini Live

Speech-to-speech in one model. WebRTC for 60-120ms latency. Voice agents.

> 📦 **Info**
>
> 🎤 Real-time audio streaming APIs
> - **OpenAI Realtime API (GA 2025):** Speech-to-speech in one model (no STT→LLM→TTS chain). WebRTC for browsers (~60-120ms latency), WebSocket for servers, SIP for telephony. Ephemeral keys for auth. Supports function calling mid-conversation.
> - **Google Gemini Live API:** Native audio processing in a single model. 30 HD voices, 24 languages. `gemini-live-2.5-flash-native-audio` optimized for low-latency dialogue (the -preview variant was deprecated 2026-03-19). Processes raw audio, not text transcriptions.
> - **Cascaded alternative:** STT (Deepgram ~200ms) → LLM (~500ms) → TTS (ElevenLabs ~150ms) = ~1000ms total. More auditable but higher latency. Use LiveKit/Pipecat for orchestration.

#### 💡 What just happened

What just happened? Real-time audio APIs represent the next frontier of streaming - **speech-to-speech without intermediate text**. OpenAI Realtime API handles audio natively via WebRTC with 60-120ms latency. Gemini Live processes raw audio in a single model. Both eliminate the STT→LLM→TTS pipeline's ~1000ms total latency. **Tradeoff:** speech-to-speech is faster but less auditable (no text transcript to inspect). For regulated industries, cascaded pipelines remain the default. For consumer voice products, native audio APIs are now production-ready. This step is only the trailer: turn-taking, interruption handling, and voice-agent latency budgets are Lesson 17.3 (Voice & Realtime Agents) - everything there builds on the streaming transport you just learned.

**📝 `09_realtime_audio.py`** - *Realtime Audio*

In [ ]:
# Minimal OpenAI Realtime API sketch - speech-to-speech over one WebSocket
import asyncio, json, os
import websockets   # pip install websockets

URL = "wss://api.openai.com/v1/realtime?model=gpt-realtime"
HEADERS = {"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"}

async def main():
    try:
        async with websockets.connect(
                URL, additional_headers=HEADERS, open_timeout=30) as ws:
            await ws.send(json.dumps({
                "type": "session.update",
                "session": {"modalities": ["audio", "text"],
                            "voice": "alloy"},
            }))
            async for raw in ws:
                event = json.loads(raw)
                if event["type"] == "response.audio.delta":
                    play(event["delta"])   # base64 PCM chunk out to the speaker
    except Exception as e:
        print(f"realtime session failed: {e}")

asyncio.run(main())
# Output: session.updated -> response.audio.delta frames stream back as speech.

---

## 🎯 Concept 10: India Streaming - Regional Latency, Mobile Resilience, Messaging

### India Streaming - Regional Latency, Mobile Resilience, Messaging

200-250ms US penalty. India-region endpoints: 21ms. Mobile: SSE works on 200Kbps.

| Endpoint | RTT from India | TTFB Impact | Strategy |
|---|---|---|---|
| US East (OpenAI/Anthropic direct) | 200-250ms | +200-250ms | Acceptable for non-critical |
| Singapore | 50-80ms | +50-80ms | Good middle ground |
| Azure Central India (Pune) | ~21ms | Minimal | GPT-4o, GPT-4 |
| AWS Bedrock Mumbai | ~15-20ms | Minimal | Claude, Llama, Mistral |
| Sarvam AI (India-native) | ~10-15ms | Minimal | Sarvam-105B, Hindi/Tamil |

> 💡 **Info**
>
> 📱 Mobile streaming for 600M+ Indian smartphone users
> - **Bandwidth:** LLM text streaming needs only ~1-5 KB/s. Even 200Kbps connections handle SSE fine. Bottleneck is latency, not throughput.
> - **India mobile:** Median 131 Mbps (5G), but 4G averages 14-17 Mbps. Rural areas: 200-500 Kbps.
> - **Resilience:** Implement Last-Event-ID for reconnection. Cache partial responses in localStorage. Gzip SSE streams (~70% savings). Batch 3-5 tokens per SSE event.
> - **WhatsApp:** No streaming support - send complete response after generation. 500M+ Indian users.
> - **Telegram:** Real streaming via `sendMessageDraft` (introduced Bot API 9.3, Dec 2025; GA for all bots since 9.5, Mar 2026). grammY stream plugin handles rate limiting.

#### 💡 What just happened

What just happened? India-region endpoints eliminate the 200-250ms US penalty entirely. **Azure Central India** provides GPT-4o at ~21ms RTT. **Bedrock Mumbai** provides Claude at ~15-20ms. **Sarvam AI** (India-native) offers the lowest latency for Indic languages with an OpenAI-compatible streaming API using 80% fewer tokens for Hindi/Tamil. Mobile streaming works even on slow 4G because text tokens need minimal bandwidth. **WhatsApp's 500M Indian users can't stream** - design for complete-response delivery. Telegram supports real streaming via sendMessageDraft (GA for all bots since Bot API 9.5).

## Key Takeaways

> ✅ **Info**
>
> What you learned
> - **Streaming = perceived speed.** TTFT 200ms vs full response 5s. Same total time, 25x better UX.
> - **Three patterns:** SSE (simplest), WebSocket (bidirectional), Chunked (universal).
> - **FastAPI SSE:** `StreamingResponse` + `text/event-stream` + `X-Accel-Buffering: no`.
> - **TTFT** is the metric. Optimize for first token, not total time.
> - **Cloud Run:** `--no-cpu-throttling` + `--timeout 600` + Secret Manager + no-buffering = production streaming.
> - **Provider formats differ** (deltas vs event lifecycle vs cumulative chunks) - and usage hides in different places; the cancelled stream is your billing blind spot (Lesson 7.3).
> - **Proxies are the #1 failure:** nginx and Cloudflare buffer by default; diagnose from the timing fingerprint, fix with `proxy_buffering off` / `X-Accel-Buffering: no`, keep idle streams alive with heartbeats.
> - **Last mile decides UX:** WhatsApp cannot stream (send complete responses, 4,096-char text limit); Telegram can (sendMessageDraft); India-region endpoints erase the ~200ms US round-trip penalty.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Streaming & Real-TimePatterns**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-7_2.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-7.2.html` - regenerate this notebook whenever the source HTML is updated.*
